# V2 Model Training

Train all 6 v2 model/feature-set combinations via walk-forward backtest,
calibrate with IsotonicRegression, produce 2025 predictions, and persist
artifacts as joblib bundles with metadata JSON.

Models: LR, RF, XGBoost x Feature Sets: TEAM_ONLY (9 cols), SP_ENHANCED_PRUNED (17 cols)

In [ ]:
import os
os.chdir(os.path.join(os.path.dirname(os.path.abspath('.')), os.path.basename(os.path.abspath('.'))))
# Ensure we are at project root
import pandas as pd
df = pd.read_parquet("data/features/feature_store_v2.parquet")
print(f"Loaded: {df.shape[0]} games, {df.shape[1]} columns, seasons {sorted(df['season'].unique())}")

In [ ]:
from src.models.backtest import run_all_v2_models
results_v2, artifacts = run_all_v2_models(df)
results_v2.to_parquet("data/results/backtest_results_v2.parquet", index=False)
print(f"Backtest results: {results_v2.shape[0]} rows")
print(f"Model/feature combos:")
print(results_v2.groupby(['model_name', 'feature_set']).size())

In [ ]:
from src.models.predict import predict_2025_v2
preds_2025_v2 = predict_2025_v2(df)
preds_2025_v2.to_parquet("data/results/predictions_2025_v2.parquet", index=False)
print(f"2025 predictions: {preds_2025_v2.shape[0]} rows")
print(f"Model/feature combos:")
print(preds_2025_v2.groupby(['model_name', 'feature_set']).size())

In [ ]:
from src.models.evaluate import compute_brier_scores
brier_v2 = compute_brier_scores(results_v2)
print("V2 Brier Scores:")
print(brier_v2[brier_v2['fold_test_year'] == 'aggregate'].to_string(index=False))

In [ ]:
import joblib
import json
from pathlib import Path
from datetime import datetime

artifacts_dir = Path("models/artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

metadata = {
    "training_date": datetime.now().isoformat(),
    "v2_feature_store": "data/features/feature_store_v2.parquet",
    "models": {}
}

for artifact in artifacts:
    name = f"{artifact['model_name']}_{artifact['feature_set']}"
    joblib.dump(artifact, artifacts_dir / f"{name}.joblib")
    print(f"Saved {name}.joblib")

    # Get fold-level Brier scores from results
    model_brier = brier_v2[
        (brier_v2['model_name'] == artifact['model_name']) &
        (brier_v2['feature_set'] == artifact['feature_set']) &
        (brier_v2['fold_test_year'] != 'aggregate')
    ]
    fold_briers = dict(zip(
        model_brier['fold_test_year'].astype(str),
        model_brier['brier_score']
    ))
    agg_brier = brier_v2[
        (brier_v2['model_name'] == artifact['model_name']) &
        (brier_v2['feature_set'] == artifact['feature_set']) &
        (brier_v2['fold_test_year'] == 'aggregate')
    ]['brier_score'].values[0]

    metadata["models"][name] = {
        "feature_cols": artifact['feature_cols'],
        "fold_brier_scores": fold_briers,
        "aggregate_brier_score": float(agg_brier),
        "calibration_method": "IsotonicRegression",
    }

with open(artifacts_dir / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(f"\nSaved model_metadata.json with {len(metadata['models'])} models")

In [ ]:
# Verify round-trip
for name in metadata["models"]:
    loaded = joblib.load(artifacts_dir / f"{name}.joblib")
    assert 'model' in loaded
    assert 'calibrator' in loaded
    assert 'feature_cols' in loaded
    print(f"  {name}: model={type(loaded['model']).__name__}, calibrator={type(loaded['calibrator']).__name__}, features={len(loaded['feature_cols'])}")
print("All 6 artifacts verified!")